In [59]:
import numpy as np
import pandas as pd
import os
import time
import pprint
import itertools
import torch

from math import ceil, sqrt
from tqdm.auto import tqdm

from sklearn.base import is_regressor
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, ParameterGrid
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, log_loss
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV

from sklearn.svm import LinearSVC
from joblib import Parallel, delayed

import pyagrum as gum
from pyagrum.skbn import BNClassifier
from pyagrum.lib.discreteTypeProcessor import DiscreteTypeProcessor

In [2]:
train = pd.read_parquet(os.path.join("..", "out", "processed", "train.parquet"))
val = pd.read_parquet(os.path.join("..", "out", "processed", "val.parquet"))
test = pd.read_parquet(os.path.join("..", "out", "processed", "test.parquet"))

meta_val = val[["open_time"]].copy()
meta_test = test[["open_time"]].copy()

train = train.drop(columns=["open_time"])
val = val.drop(columns=["open_time"])
test = test.drop(columns=["open_time"])

X_train = train.drop(columns=["label"])
y_train = train["label"]

X_val = val.drop(columns=["label"])
y_val = val["label"]

X_test = test.drop(columns=["label"])
y_test = test["label"]

rng = np.random.RandomState(0)
train_idx = rng.choice(len(X_train), 200_000, replace=False)
valid_idx = rng.choice(len(X_val), 30_000, replace=False)

X_train_sample = X_train.iloc[train_idx].to_numpy()
y_train_sample = y_train.iloc[train_idx].to_numpy()
X_val_sample = X_val.iloc[valid_idx].to_numpy()
y_val_sample = y_val.iloc[valid_idx].to_numpy()

print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)

print(X_train.describe().loc[["mean", "std"]].T)

Training set shape: (1127670, 36) (1127670,)
Validation set shape: (188886, 36) (188886,)
Test set shape: (281294, 36) (281294,)
                               mean       std
symbol=BTCUSDT         8.404675e-02  0.277458
symbol=ETHUSDT         9.571062e-02  0.294194
symbol=BNBUSDT         9.720840e-02  0.296241
symbol=XRPUSDT         1.029530e-01  0.303898
symbol=ADAUSDT         1.097883e-01  0.312626
symbol=DOGEUSDT        9.485488e-02  0.293015
symbol=SOLUSDT         1.016264e-01  0.302156
symbol=LTCUSDT         1.044862e-01  0.305890
symbol=LINKUSDT        1.138782e-01  0.317663
symbol=AVAXUSDT        9.544725e-02  0.293832
returns                4.839153e-18  1.000000
volatility            -2.580882e-17  1.000000
upper_wick            -6.452205e-17  1.000000
lower_wick             0.000000e+00  1.000000
body_ratio             4.375401e-17  1.000000
taker_buy_ratio       -2.790579e-16  1.000000
taker_buy_ratio_16    -1.451746e-15  1.000000
relative_volume       -1.729997e-16  1.0000

In [3]:
model_results = []

def get_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    elif hasattr(model, "decision_function"):
        return model.decision_function(X)
    return model.predict(X)

def evaluate_model(name, model, X_train, y_train, X_val, y_val):
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    fit_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    raw = model.predict(X_val)
    predict_time = time.perf_counter() - t0

    reg = is_regressor(model)
    y_pred = (raw > 0.5).astype(int) if reg else raw
    scores = raw if reg else get_scores(model, X_val)

    row = {
        "model": name,
        "fit_time": fit_time,
        "predict_time": predict_time,
        "accuracy": accuracy_score(y_val, y_pred),
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "recall": recall_score(y_val, y_pred, zero_division=0),
        "f1": f1_score(y_val, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_val, scores),
        "confusion_matrix": confusion_matrix(y_val, y_pred).tolist(),
        "log_loss": log_loss(y_val, scores),
    }   
    model_results.append(row)
    pprint.pprint(row)


In [76]:
# ----------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------
def run_parallel(fn, combos, desc, n_jobs=10):
    """Run fn(*combo) for every combo in parallel with a progress bar that vanishes at the end.
    Needs joblib >= 1.3 for return_as="generator"."""
    gen = Parallel(n_jobs=n_jobs, return_as="generator_unordered")(
        delayed(fn)(*c) for c in combos
    )
    return list(tqdm(gen, total=len(combos), desc=desc, leave=False))

def run_serial(fn, combos, desc):
    return [fn(*c) for c in tqdm(combos, desc=desc, leave=False)]

def to_df(rows, by="val_log_loss", ascending=True):
    return pd.DataFrame(rows).sort_values(by=by, ascending=ascending).reset_index(drop=True)

def metrics(y_true, p):
    """p = probability of the positive class."""
    return {
        "log_loss": log_loss(y_true, p),
        "auc": roc_auc_score(y_true, p),
    }


# ----------------------------------------------------------------------------
# Logistic Regression
# ----------------------------------------------------------------------------
def evaluate_logistic_regression(solver, penalty, C, max_iter=1000):
    m = LogisticRegression(
        solver=solver,
        penalty=penalty,
        C=C,
        max_iter=max_iter,
        random_state=42,
    )
    m.fit(X_train, y_train)
    p_train = m.predict_proba(X_train)[:, 1]
    p_val = m.predict_proba(X_val)[:, 1]
    return {
        "solver": solver,
        "penalty": penalty,
        "C": C,
        "train_log_loss": log_loss(y_train, p_train),
        "val_log_loss": log_loss(y_val, p_val),
        "train_auc": roc_auc_score(y_train, p_train),
        "val_auc": roc_auc_score(y_val, p_val),
    }


def select_logistic_regression(solvers, penalties, Cs):
    combos = list(itertools.product(solvers, penalties, Cs))
    return to_df(run_parallel(evaluate_logistic_regression, combos, "LogReg"))


# ----------------------------------------------------------------------------
# KNN
# ----------------------------------------------------------------------------
def evaluate_knn(n_neighbors, weights, algorithm, metric, p):
    m = KNeighborsClassifier(
        n_neighbors=n_neighbors,
        weights=weights,
        algorithm=algorithm,
        metric=metric,
        p=p,
        batch_size=500,
    )
    m.fit(X_train_sample, y_train_sample)
    p_train = m.predict_proba(X_train_sample)[:, 1]
    p_val = m.predict_proba(X_val_sample)[:, 1]
    return {
        "n_neighbors": n_neighbors,
        "weights": weights,
        "algorithm": algorithm,
        "metric": metric,
        "p": p,
        "train_log_loss": log_loss(y_train_sample, p_train),
        "val_log_loss": log_loss(y_val_sample, p_val),
        "train_auc": roc_auc_score(y_train_sample, p_train),
        "val_auc": roc_auc_score(y_val_sample, p_val),
    }


def select_knn(n_neighbors, weights, algorithms, metrics_, ps):
    combos = list(itertools.product(n_neighbors, weights, algorithms, metrics_, ps))
    return to_df(run_serial(evaluate_knn, combos, "KNN"))


# ----------------------------------------------------------------------------
# Decision Tree
# ----------------------------------------------------------------------------
def evaluate_tree(criterion, max_depth, min_samples_leaf, splitter="best",
                  min_samples_split=2, max_leaf_nodes=None, min_impurity_decrease=0.0):
    m = DecisionTreeClassifier(
        criterion=criterion,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        splitter=splitter,
        min_samples_split=min_samples_split,
        max_leaf_nodes=max_leaf_nodes,
        min_impurity_decrease=min_impurity_decrease,
        random_state=42,
    )
    m.fit(X_train, y_train)
    p_train = m.predict_proba(X_train)[:, 1]
    p_val = m.predict_proba(X_val)[:, 1]
    return {
        "criterion": criterion,
        "splitter": splitter,
        "max_depth": max_depth,
        "min_samples_leaf": min_samples_leaf,
        "min_samples_split": min_samples_split,
        "max_leaf_nodes": max_leaf_nodes,
        "min_impurity_decrease": min_impurity_decrease,
        "train_log_loss": log_loss(y_train, p_train),
        "val_log_loss": log_loss(y_val, p_val),
        "train_auc": roc_auc_score(y_train, p_train),
        "val_auc": roc_auc_score(y_val, p_val),
    }


def select_tree(criterions, max_depths, min_samples_leaves,
                splitters=("best", "random"), min_samples_split=(2, 5, 10),
                max_leaf_nodes=(None,), min_impurity_decrease=(0.0,)):
    combos = list(itertools.product(
        criterions, max_depths, min_samples_leaves,
        splitters, min_samples_split, max_leaf_nodes, min_impurity_decrease,
    ))
    return to_df(run_parallel(evaluate_tree, combos, "Tree"))


# ----------------------------------------------------------------------------
# Gaussian Naive Bayes
# ----------------------------------------------------------------------------
def evaluate_gaussian_nb(var_smoothing):
    m = GaussianNB(var_smoothing=var_smoothing)
    m.fit(X_train, y_train)
    p_train = m.predict_proba(X_train)[:, 1]
    p_val = m.predict_proba(X_val)[:, 1]
    return {
        "var_smoothing": var_smoothing,
        "train_log_loss": log_loss(y_train, p_train),
        "val_log_loss": log_loss(y_val, p_val),
        "train_auc": roc_auc_score(y_train, p_train),
        "val_auc": roc_auc_score(y_val, p_val),
    }


def select_gaussian_nb(var_smoothing):
    combos = [(v,) for v in var_smoothing]
    return to_df(run_parallel(evaluate_gaussian_nb, combos, "GaussianNB"))


# ----------------------------------------------------------------------------
# Random Forest
# ----------------------------------------------------------------------------
def evaluate_random_forest(n_estimators, max_features, max_depth=None):
    m = RandomForestClassifier(
        n_estimators=n_estimators,
        max_features=max_features,
        max_depth=max_depth,
        random_state=42,
    )
    m.fit(X_train, y_train)
    p_train = m.predict_proba(X_train)[:, 1]
    p_val = m.predict_proba(X_val)[:, 1]
    return {
        "n_estimators": n_estimators,
        "max_features": max_features,
        "max_depth": max_depth,
        "train_log_loss": log_loss(y_train, p_train),
        "val_log_loss": log_loss(y_val, p_val),
        "train_auc": roc_auc_score(y_train, p_train),
        "val_auc": roc_auc_score(y_val, p_val),
    }


def select_random_forest(n_estimators, max_features, max_depths=(None,)):
    combos = list(itertools.product(n_estimators, max_features, max_depths))
    rows = run_parallel(evaluate_random_forest, combos, "RandomForest", n_jobs=4)
    return to_df(rows)


# ----------------------------------------------------------------------------
# AdaBoost
# ----------------------------------------------------------------------------
def adaboost_evaluate(estimator, n_estimators, learning_rate):
    m = AdaBoostClassifier(estimator=estimator, n_estimators=n_estimators, learning_rate=learning_rate, random_state=42)
    m.fit(X_train, y_train)
    p_train = m.predict_proba(X_train)[:, 1]
    p_val = m.predict_proba(X_val)[:, 1]
    return {
        "n_estimators": n_estimators,
        "learning_rate": learning_rate,
        "train_log_loss": log_loss(y_train, p_train),
        "val_log_loss": log_loss(y_val, p_val),
        "train_auc": roc_auc_score(y_train, p_train),
        "val_auc": roc_auc_score(y_val, p_val),
    }




def adaboost_select(estimators, n_estimators, learning_rates):
    combos = list(itertools.product(estimators, n_estimators, learning_rates))
    return to_df(run_parallel(adaboost_evaluate, combos, "AdaBoost"))


# ----------------------------------------------------------------------------
# Linear SVM
# ----------------------------------------------------------------------------

def evaluate_svm(C, penalty, loss, dual, tol, class_weight=None, max_iter=1000, random_state=42):
    m = LinearSVC(C=C, penalty=penalty, loss=loss, dual=dual,
                  max_iter=max_iter, random_state=random_state, tol=tol, class_weight=class_weight)
    m = CalibratedClassifierCV(m, method="sigmoid", cv=3)
    m.fit(X_train, y_train)
    s_train = m.predict_proba(X_train)[:, 1]
    s_val = m.predict_proba(X_val)[:, 1]
    return {
        "C": C,
        "penalty": penalty,
        "loss": loss,
        "dual": dual,
        "train_log_loss": log_loss(y_train, s_train),
        "val_log_loss": log_loss(y_val, s_val),
        "train_auc": roc_auc_score(y_train, s_train),
        "tol": tol,
        "class_weight": class_weight,
        "val_auc": roc_auc_score(y_val, s_val),
    }


def select_svm(Cs, penalties, losses, duals, tols):
    combos = list(itertools.product(Cs, penalties, losses, duals, tols))
    return to_df(run_parallel(evaluate_svm, combos, "SVM"))


# ----------------------------------------------------------------------------
# MLP
# ----------------------------------------------------------------------------
def evaluate_mlp(hidden_layer_sizes, activation, solver, alpha, learning_rate, learning_rate_init, max_iter=1000):
    m = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        activation=activation,
        solver=solver,
        alpha=alpha,
        learning_rate=learning_rate,
        learning_rate_init=learning_rate_init,
        max_iter=max_iter,
        random_state=42,
    )
    m.fit(X_train, y_train)
    p_train = m.predict_proba(X_train)[:, 1]
    p_val = m.predict_proba(X_val)[:, 1]
    return {
        "hidden_layer_sizes": hidden_layer_sizes,
        "activation": activation,
        "solver": solver,
        "alpha": alpha,
        "learning_rate_init": learning_rate_init,
        "learning_rate": learning_rate,
        "train_log_loss": log_loss(y_train, p_train),
        "val_log_loss": log_loss(y_val, p_val),
        "train_auc": roc_auc_score(y_train, p_train),
        "val_auc": roc_auc_score(y_val, p_val),
    }


def select_mlp(hidden_layer_sizes_list, activations, solvers, alphas, learning_rates, learning_rate_inits=(0.001,)):
    combos = list(itertools.product(hidden_layer_sizes_list, activations, solvers, alphas, learning_rates, learning_rate_inits))
    return to_df(run_parallel(evaluate_mlp, combos, "MLP"))

# Majority Classifier

In [77]:
evaluate_model('Majority', DummyClassifier(strategy='most_frequent'), X_train, y_train, X_val, y_val)

{'accuracy': 0.5062100949779232,
 'confusion_matrix': [[0, 93270], [0, 95616]],
 'f1': 0.6721639918172807,
 'fit_time': 0.07768809999834048,
 'log_loss': 17.797992183660813,
 'model': 'Majority',
 'precision': 0.5062100949779232,
 'predict_time': 0.002443400000629481,
 'recall': 1.0,
 'roc_auc': 0.5}


# Linear Regression

In [78]:
evaluate_model('Linear Regression', LinearRegression(), X_train, y_train, X_val, y_val)

{'accuracy': 0.5270533549336637,
 'confusion_matrix': [[43584, 49686], [39647, 55969]],
 'f1': 0.556155630965216,
 'fit_time': 1.2410286999947857,
 'log_loss': 0.6908403255373232,
 'model': 'Linear Regression',
 'precision': 0.529733566797596,
 'predict_time': 0.11835579999751644,
 'recall': 0.5853518239625167,
 'roc_auc': 0.5378091515753877}


# Logistic Regression

In [80]:
results = select_logistic_regression(
    solvers=['lbfgs', 'newton-cg', 'sag'],
    penalties=['l2', None],
    Cs=[0.00001, 0.0001, 0.001, 0.01]
)

print(results.sort_values(by="val_log_loss", ascending=True).head(5))

results = select_logistic_regression(
    solvers=['liblinear'],
    penalties=['l1', 'l2'],
    Cs=[0.00001, 0.0001, 0.0001, 0.001, 0.01]
)

print(results.sort_values(by="val_log_loss", ascending=True).head(5))

results = select_logistic_regression(
    solvers=['saga'],
    penalties=['l1', 'l2', 'elasticnet', None],
    Cs=[0.00001, 0.0001, 0.0001, 0.001, 0.01]
)

print(results.sort_values(by="val_log_loss", ascending=True).head(5))

      solver penalty       C  train_log_loss  val_log_loss  train_auc  \
0      lbfgs      l2  0.0001        0.689594      0.690824   0.548465   
1  newton-cg      l2  0.0001        0.689593      0.690828   0.548459   
2        sag      l2  0.0001        0.689593      0.690829   0.548458   
3  newton-cg      l2  0.0010        0.689571      0.690831   0.548451   
4        sag      l2  0.0010        0.689571      0.690832   0.548452   

    val_auc  
0  0.537722  
1  0.537700  
2  0.537699  
3  0.537846  
4  0.537845  


      solver penalty       C  train_log_loss  val_log_loss  train_auc  \
0  liblinear      l1  0.0100        0.689579      0.690825   0.548443   
1  liblinear      l2  0.0001        0.689593      0.690828   0.548458   
2  liblinear      l2  0.0001        0.689593      0.690828   0.548458   
3  liblinear      l2  0.0010        0.689571      0.690832   0.548452   
4  liblinear      l2  0.0100        0.689568      0.690839   0.548452   

    val_auc  
0  0.537821  
1  0.537697  
2  0.537697  
3  0.537845  
4  0.537859  


  solver     penalty       C  train_log_loss  val_log_loss  train_auc  \
0   saga          l1  0.0100        0.689579      0.690828   0.548444   
1   saga          l2  0.0001        0.689593      0.690829   0.548458   
2   saga  elasticnet  0.0001        0.689593      0.690829   0.548458   
3   saga          l2  0.0001        0.689593      0.690829   0.548458   
4   saga  elasticnet  0.0001        0.689593      0.690829   0.548458   

    val_auc  
0  0.537802  
1  0.537699  
2  0.537699  
3  0.537699  
4  0.537699  


In [81]:
# evaluate_model("Logistic Regression", LogisticRegression(solver='lbfgs', penalty='l2', C=0.0001), X_train, y_train, X_val, y_val)

# KNN

In [82]:
# ks = [2200, 2300, 2400, 2500, 2600, 2800,]

# t0 = time.perf_counter()
# nn = NearestNeighbors(n_neighbors=max(ks), n_jobs=-1).fit(X_train_sample)
# distances, indices = nn.kneighbors(X_val_sample)

# neighbour_labels = y_train_sample[indices]

# rows = []
# for k in ks:
#     scores = neighbour_labels[:, :k].mean(axis=1)
#     y_pred = (scores > 0.5).astype(int)
#     row = {
#         'k': k,
#         'accuracy': accuracy_score(y_val_sample, y_pred),
#         'roc_auc_score': roc_auc_score(y_val_sample, scores),
#         'log_loss': log_loss(y_val_sample, scores)
#     }
#     rows.append(row)
# knn_results = pd.DataFrame(rows).sort_values(by="log_loss", ascending=True).reset_index(drop=True)
# print(knn_results.head(10))

In [83]:
# results = select_knn(
#     n_neighbors=[2500],
#     weights=['uniform', 'distance'],
#     algorithms=['brute'],
#     metrics=['euclidean', 'manhattan', 'minkowski'],
#     ps=[1, 2]
# )

# print(results.head(5))

In [84]:
# evaluate_model('3501-NN', KNeighborsClassifier(n_neighbors=3501, n_jobs=-1), X_train_sample, y_train_sample, X_val_sample, y_val_sample)

# Decision Tree

In [85]:
results = select_tree(
   criterions=['entropy', 'gini', 'log_loss'],
   max_depths=[4, 5, 6],
   min_samples_leaves=[2000, 3000],
   splitters=['best', 'random'],
   min_samples_split=[2, 24, 48, 200, 10000],
   max_leaf_nodes=[None],
   min_impurity_decrease=[0.0, 0.1]
   )

print(results.sort_values(by="val_log_loss", ascending=True).head(10))

  criterion splitter  max_depth  min_samples_leaf  min_samples_split  \
0   entropy     best          4              2000                 48   
1   entropy     best          4              2000                200   
2   entropy     best          4              2000                  2   
3   entropy     best          4              2000                 24   
4   entropy     best          4              3000                  2   
5   entropy     best          4              3000                 24   
6   entropy     best          4              3000                 48   
7   entropy     best          4              3000                200   
8  log_loss     best          4              3000                 48   
9  log_loss     best          4              3000                200   

  max_leaf_nodes  min_impurity_decrease  train_log_loss  val_log_loss  \
0           None                    0.0        0.688985      0.691937   
1           None                    0.0        0.688985      

In [86]:
# evaluate_model('ID3', DecisionTreeClassifier(criterion='entropy', splitter='best', max_depth=4, min_samples_leaf=20000, min_samples_split=48, max_leaf_nodes=None, min_impurity_decrease=0.0, random_state=42), X_train, y_train, X_val, y_val)

# Random Forest

In [88]:
results = select_random_forest(
    n_estimators=[100, 200, 400],
    max_features=['sqrt', 'log2', 0.1, 0.2, 0.3, 1.0, 2],
    max_depths=[6, 7, 8] 
)

print(results.sort_values(by="val_log_loss", ascending=True).head(10))

   n_estimators max_features  max_depth  train_log_loss  val_log_loss  \
0           100         sqrt          8        0.684907      0.690679   
1           400         sqrt          8        0.684849      0.690691   
2           400         log2          8        0.685211      0.690693   
3           200          0.2          8        0.684534      0.690695   
4           200         sqrt          8        0.684858      0.690701   
5           400          0.2          8        0.684562      0.690702   
6           100          0.2          8        0.684520      0.690706   
7           400          0.1          8        0.686291      0.690707   
8           200          0.1          8        0.686266      0.690708   
9           200         log2          8        0.685195      0.690712   

   train_auc   val_auc  
0   0.576768  0.538751  
1   0.577330  0.538658  
2   0.575907  0.538575  
3   0.578509  0.538842  
4   0.577240  0.538590  
5   0.578444  0.538739  
6   0.578594  0.53875

In [ ]:
evaluate_model('Random Forest', RandomForestClassifier(n_estimators=100, max_features='log2', max_depth=6, random_state=42), X_train, y_train, X_val, y_val)

# Naive Bayes

In [89]:
results = select_gaussian_nb(
    var_smoothing=[1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 20, 50, 100, 200]
)

print(results.sort_values(by="val_log_loss", ascending=True).head(10))

   var_smoothing  train_log_loss  val_log_loss  train_auc   val_auc
0        10.0000        0.691586      0.692103   0.546570  0.535133
1        20.0000        0.692130      0.692422   0.546591  0.534966
2        50.0000        0.692634      0.692766   0.546564  0.534894
3       100.0000        0.692834      0.692910   0.546550  0.534880
4       200.0000        0.692941      0.692989   0.546542  0.534876
5         1.0000        0.712818      0.705514   0.544433  0.534740
6         0.1000        0.802785      0.757545   0.541794  0.532754
7         0.0100        0.829269      0.773140   0.541303  0.532313
8         0.0010        0.832453      0.775202   0.541113  0.532172
9         0.0001        0.832778      0.775416   0.541088  0.532153


In [ ]:
# evaluate_model("Naive Bayes", GaussianNB(var_smoothing=10), X_train, y_train, X_val, y_val)

{'accuracy': 0.5162743665491355,
 'confusion_matrix': [[72061, 21209], [70160, 25456]],
 'f1': 0.35782711676190077,
 'fit_time': 0.7664935000002515,
 'log_loss': 0.6921025471995634,
 'model': 'Naive Bayes',
 'precision': 0.5455051966141647,
 'predict_time': 0.10994399999981397,
 'recall': 0.26623159303882193,
 'roc_auc': 0.5351330642429624}


# AdaBoost

In [ ]:
results = adaboost_select(
    estimators=[DecisionTreeClassifier(max_depth=1, random_state=42), DecisionTreeClassifier(max_depth=2, random_state=42)],
    n_estimators=[400, 800, 1000, 1500],
    learning_rates=[0.1, 0.3, 0.5, 1.0]
)

print(results.sort_values(by="val_log_loss", ascending=True).head(10))

AdaBoost:   0%|          | 0/32 [00:00<?, ?it/s]

# SVM

In [ ]:
results = select_svm(
    Cs=[0.01, 0.1, 1.0, 10.0, 100.0], 
    penalties=['l2'], 
    losses=['squared_hinge'], 
    duals=[True], 
    tols=[1e-4, 1e-3, 1e-2, 1e-1]
)
print(results.sort_values(by="val_log_loss", ascending=True).head(10))

KeyboardInterrupt: 

# Neural Network

In [ ]:
results = select_mlp(
    hidden_layer_sizes_list=[(64,), (64, 32), (128, 64)],
    activations=["relu"],
    solvers=["sgd"],
    alphas=[1e-4, 1e-2],
    learning_rates=["constant", "invscaling", "adaptive"],
    learning_rate_inits=(0.001, 0.01)
)

print(results.head(10))

results = select_mlp(
    hidden_layer_sizes_list=[(64,), (64, 32), (128, 64)],
    activations=["relu"],
    solvers=["adam", "lbfgs"],
    alphas=[1e-4, 1e-2],
    learning_rates=["constant"],
    learning_rate_inits=(0.001,),
)

print(results.sort_values(by="val_log_loss", ascending=True).head(10))

KeyboardInterrupt: 